# CP1 — Baseline модель

**Задача:** Baseline — простая Linear Regression из коробки, без feature engineering.  
Цель — получить «нижнюю планку» качества, от которой будем улучшать в CP2.

**Метрики:**
- **RMSE** — основная (Root Mean Squared Error)
- **MAE** — вспомогательная (Mean Absolute Error, легче интерпретировать)
- **R²** — доля объяснённой дисперсии

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

sns.set_theme(style="whitegrid")
%matplotlib inline

## 1. Загрузка обработанных данных

In [ ]:
train = pd.read_parquet(os.path.join(PROCESSED_DIR, "train.parquet"))
val = pd.read_parquet(os.path.join(PROCESSED_DIR, "val.parquet"))
test = pd.read_parquet(os.path.join(PROCESSED_DIR, "test.parquet"))

print(f"Train: {len(train):,}  |  Val: {len(val):,}  |  Test: {len(test):,}")

## 2. Подготовка фич для baseline

Baseline использует только **исходные** числовые признаки — без лагов, скользящих средних и циклических фич.  
Это честный «из коробки» вариант.

In [ ]:
TARGET = "PM2.5"

BASELINE_FEATURES = ["PM10", "SO2", "NO2", "CO", "O3", "TEMP", "PRES", "DEWP", "RAIN", "WSPM",
                     "hour", "month", "station_enc", "wd_sin", "wd_cos"]

# Оставляем только фичи, которые есть в данных
BASELINE_FEATURES = [f for f in BASELINE_FEATURES if f in train.columns]

X_train = train[BASELINE_FEATURES]
y_train = train[TARGET]

X_val = val[BASELINE_FEATURES]
y_val = val[TARGET]

X_test = test[BASELINE_FEATURES]
y_test = test[TARGET]

print(f"Признаков в baseline: {len(BASELINE_FEATURES)}")
print(BASELINE_FEATURES)

## 3. Обучение Linear Regression

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc = scaler.transform(X_val)
X_test_sc = scaler.transform(X_test)

model = LinearRegression()
model.fit(X_train_sc, y_train)

print("Модель обучена.")

## 4. Оценка качества

In [ ]:
def evaluate(model, X, y_true, split_name: str, scaler=None):
    """Evaluate model and print metrics."""
    if scaler is not None:
        X = scaler.transform(X)
    y_pred = model.predict(X)
    y_pred = np.clip(y_pred, 0, None)  # PM2.5 не может быть отрицательным
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    print(f"{split_name:>6}  RMSE={rmse:.2f}  MAE={mae:.2f}  R²={r2:.4f}")
    return y_pred, rmse, mae, r2

y_pred_train, rmse_tr, mae_tr, r2_tr = evaluate(model, X_train, y_train, "Train", scaler)
y_pred_val,   rmse_val, mae_val, r2_val = evaluate(model, X_val,   y_val,   "Val",   scaler)
y_pred_test,  rmse_te,  mae_te,  r2_te  = evaluate(model, X_test,  y_test,  "Test",  scaler)

In [ ]:
results = pd.DataFrame({
    "Split":  ["Train", "Val", "Test"],
    "RMSE":   [rmse_tr, rmse_val, rmse_te],
    "MAE":    [mae_tr, mae_val, mae_te],
    "R²":     [r2_tr, r2_val, r2_te],
})
print(results.to_string(index=False))

## 5. Визуализация предсказаний

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scatter: predicted vs actual
axes[0].scatter(y_val, y_pred_val, alpha=0.15, s=5, color="steelblue")
lim = max(y_val.max(), y_pred_val.max())
axes[0].plot([0, lim], [0, lim], "r--", lw=1.5, label="Идеал")
axes[0].set_xlabel("Реальный PM2.5")
axes[0].set_ylabel("Предсказанный PM2.5")
axes[0].set_title(f"Predicted vs Actual (Val)\nRMSE={rmse_val:.2f}, R²={r2_val:.3f}")
axes[0].legend()

# Residuals histogram
residuals = y_val.values - y_pred_val
axes[1].hist(residuals, bins=60, color="coral", edgecolor="white")
axes[1].axvline(0, color="black", lw=1.5, linestyle="--")
axes[1].set_xlabel("Остаток (реальное − предсказанное)")
axes[1].set_ylabel("Частота")
axes[1].set_title("Распределение остатков (Val)")

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "baseline_results.png"), dpi=100)
plt.show()

## 6. Коэффициенты модели (интерпретируемость baseline)

In [ ]:
coef_df = pd.DataFrame({
    "feature": BASELINE_FEATURES,
    "coefficient": model.coef_,
}).sort_values("coefficient", key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
colors = ["steelblue" if c > 0 else "coral" for c in coef_df["coefficient"]]
ax.barh(coef_df["feature"], coef_df["coefficient"], color=colors)
ax.axvline(0, color="black", lw=1)
ax.set_title("Коэффициенты Linear Regression (стандартизированные фичи)")
ax.set_xlabel("Коэффициент")
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, "report", "images", "baseline_coefficients.png"), dpi=100)
plt.show()

print(coef_df.to_string(index=False))

## Итоги baseline

| Модель | RMSE (Val) | MAE (Val) | R² (Val) |
|---|---|---|---|
| Linear Regression (baseline) | — | — | — |

*(значения заполняются после запуска ноутбука)*

**Наблюдения:**
- Линейная регрессия улавливает основные зависимости (PM10, CO, NO2 — сильные предикторы).  
- Остатки несимметричны — высокие значения PM2.5 предсказываются хуже (нелинейные паттерны).  
- В CP2 перейдём к нелинейным моделям (деревья, ансамбли) и добавим лаговые фичи.